In [ ]:
# Per leggere i file
import gzip
import pickle

import numpy as np
from sktime.clustering.k_means import TimeSeriesKMeans
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

import matplotlib.pyplot as plt
import pandas as pd

## Caricamento Dati
---

In [ ]:
with gzip.open("ts_transformed_paa.pkl.gz", "rb") as f:
    ts_transformed_paa = pickle.load(f)

# id non serve per il clustering, anzi può essere dannoso
for df in ts_transformed_paa:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)


colonne_fisse = ['weekday', 'quarter',
                 'relative_date_PCIAT', 'sii_binary']

In [ ]:
df_globale = pd.concat(ts_transformed_paa, ignore_index=True)

# Opzionale: campionamento per velocizzare la generazione grafica in caso di dataset massivi.
# Se si desidera analizzare il 100% dei dati, lasciare frac=1.0.
df_plot = df_globale.sample(frac=1.0, random_state=222)

# ==========================================
# 3. Visualizzazione tramite Istogrammi
# ==========================================
# Viene creata una griglia automatica in base al numero di colonne
df_plot.hist(bins=50, figsize=(15, 10), edgecolor='black',
             grid=False, alpha=0.75, color='steelblue')
plt.suptitle("Distribuzione Globale delle Variabili (Istogrammi)",
             y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## Whole Clustering
---
K-Means Euclideo è fattibile, K-Means con DTW è un mattone non runnabile I guess (da portatile)

In [ ]:
'''# Bisogna trasformare i dati in un array 3d come restituiva paa nel preprocessing
X = np.array(ts_normalized)
X_sktime = np.transpose(X, (0, 2, 1))

k = 5

# K Means Euclideo
kmeans_euclidean = TimeSeriesKMeans(
    n_clusters=k,
    metric="euclidean",
    random_state=222
)
labels_euclidean = kmeans_euclidean.fit_predict(X_sktime)
print("Clustering Euclideo completato.")

# K Means DTW
kmeans_dtw = TimeSeriesKMeans(
    n_clusters=k,
    metric="dtw",
    random_state=222
)
labels_dtw = kmeans_dtw.fit_predict(X_sktime)
print("Clustering DTW completato.")'''

## Approximate Clustering
---

In [ ]:
# Bisogna trasformare i dati in un array 3d come restituiva paa nel preprocessing
X = np.array(ts_transformed_paa)
X_sktime = np.transpose(X, (0, 2, 1))

k = 5

# K Means Euclideo

kmeans_euclidean = TimeSeriesKMeans(
    n_clusters=k,
    metric="euclidean",
    init_algorithm="kmeans++",
    n_init=10,
    random_state=222
)

labels_euclidean = kmeans_euclidean.fit_predict(X_sktime)
print("Clustering Euclideo completato.")

# K Means DTW
kmeans_dtw = TimeSeriesKMeans(
    n_clusters=k,
    metric="dtw",
    init_algorithm="kmeans++",
    n_init=10,
    random_state=222
)

labels_dtw = kmeans_dtw.fit_predict(X_sktime)
print("Clustering DTW completato.")


# Centroidi
centroidi_euclidean = kmeans_euclidean.cluster_centers_
centroidi_dtw = kmeans_dtw.cluster_centers_

In [ ]:
# ==========================================
# 1. Preparazione e appiattimento dei dati
# ==========================================
X_flattened_list = []

for df in ts_transformed_paa:
    # L'attributo .values estrae l'array NumPy dal DataFrame.
    # Il metodo .flatten() converte la matrice (es. 20x13) in un vettore 1D.
    vettore_appiattito = df.values.flatten()
    X_flattened_list.append(vettore_appiattito)

# Conversione della lista in un array NumPy bidimensionale
X_flattened = np.array(X_flattened_list)

# Definizione del numero di cluster
k_clusters = 5

# ==========================================
# 2. Clustering Gerarchico (Agglomerativo)
# ==========================================
# L'algoritmo agglomerativo procede in modo deterministico unendo i cluster
# in base alle distanze matematiche esatte.
hierarchical_clusterer = AgglomerativeClustering(
    n_clusters=k_clusters,
    metric='euclidean',
    linkage='ward'
)

labels_ger = hierarchical_clusterer.fit_predict(X_flattened)

# Centroidi
centroidi_ger_list = []

# Iterazione su ogni cluster da 0 a k-1
for cluster_id in range(k_clusters):
    # Estrazione di tutte le serie storiche appartenenti al cluster corrente
    serie_nel_cluster = X_sktime[labels_ger == cluster_id]

    # Calcolo della media lungo l'asse delle istanze (asse 0)
    centroide = np.mean(serie_nel_cluster, axis=0)
    centroidi_ger_list.append(centroide)

# Conversione della lista risultante in un array NumPy
centroidi_ger = np.array(centroidi_ger_list)

In [ ]:
import pacmap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

dict_labels = {
    "labels_euclidean": labels_euclidean,
    "labels_dtw": labels_dtw,
    "labels_ger": labels_ger
}

# PCA
pca = PCA(n_components=2, random_state=222)
X_pca = pca.fit_transform(X_flattened)

# t-SNE
tsne = TSNE(n_components=2, random_state=222)
X_tsne = tsne.fit_transform(X_flattened)

# PacMAP
# L'inizializzazione con PCA è raccomandata per PacMAP per migliorare la stabilità
pm = pacmap.PaCMAP(n_components=2, random_state=222)
X_pacmap = pm.fit_transform(X_flattened, init="pca")

# ==========================================
# 2. Visualizzazione
# ==========================================

for title_label, colors in dict_labels.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"\nGrafici: {title_label}", fontsize=16)

    # Grafico PCA
    scatter_pca = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=colors,
                                  cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
    axes[0].set_title('PCA')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')

    # Grafico t-SNE
    scatter_tsne = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors,
                                   cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
    axes[1].set_title('t-SNE')
    axes[1].set_xlabel('Dimension1')
    axes[1].set_ylabel('Dimension2')

    # Grafico PacMAP
    scatter_pacmap = axes[2].scatter(X_pacmap[:, 0], X_pacmap[:, 1],
                                     c=colors, cmap='tab10', alpha=0.6, edgecolors='w', linewidth=0.5)
    axes[2].set_title('PacMAP')
    axes[2].set_xlabel('Dimension1')
    axes[2].set_ylabel('Dimension2')

    # Aggiunta di una legenda per i cluster
    # Utilizza le informazioni dell'ultimo scatter plot per generare la legenda
    legend = fig.legend(*scatter_pacmap.legend_elements(),
                        title="Cluster", loc='center right')
    fig.subplots_adjust(right=0.92)  # Spazio per la legenda

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.show()

In [ ]:
# Valutazione centroidi

dict_centroids = {
    "K-Means Euclideo": centroidi_euclidean,
    "K-Means DTW": centroidi_dtw,
    "Clustering Gerarchico": centroidi_ger
}

nomi_colonne_mutevoli = [
    "X", "Y", "Z", "enmo", "anglez",
    "non-wear_flag", "light", "battery_voltage"
]
nomi_colonne_statiche = [
    "weekday", "quarter", "relative_date_PCIAT", "sii_binary"
]

# Unione delle liste per coprire tutte le colonne del centroide
nomi_colonne_totali = nomi_colonne_mutevoli + nomi_colonne_statiche

for nome_tecnica, centroidi in dict_centroids.items():
    for i, centroide in enumerate(centroidi):
        centroide_trasposto = centroide.T

        # Creazione del DataFrame
        df_centroide = pd.DataFrame(centroide_trasposto)

        # Assegnazione dei nomi esatti alle colonne
        df_centroide.columns = nomi_colonne_totali

        # Separazione delle colonne utilizzando i nomi definiti
        df_dinamiche = df_centroide[nomi_colonne_mutevoli]
        df_fisse = df_centroide[nomi_colonne_statiche]

        # Estrazione del singolo valore per le variabili fisse
        valori_fissi = df_fisse.iloc[0]

        # Formattazione del testo per le variabili fisse (ora usa i nomi reali)
        testo_fisse = "Valori delle variabili fisse:\n" + " | ".join(
            [f"{col}: {val:.4f}" for col, val in valori_fissi.items()]
        )

        # Creazione del grafico limitatamente alle variabili dinamiche
        ax_array = df_dinamiche.plot(
            kind='line',
            subplots=True,
            figsize=(10, 8),
            title=f"Trend delle variabili: Cluster {i} ({nome_tecnica})",
            sharex=True,
            legend=False
        )

        plt.xlabel("Time steps")

        # Ottimizzazione degli spazi, lasciando un margine inferiore per il testo
        plt.tight_layout(rect=[0, 0.08, 1, 1])

        # Inserimento della stringa di testo contenente i valori fissi
        plt.figtext(
            0.5, 0.02, testo_fisse,
            ha="center", va="bottom", fontsize=10,
            bbox={"facecolor": "white", "edgecolor": "lightgray",
                  "alpha": 0.9, "boxstyle": "round,pad=0.5"}
        )

        plt.show()

In [ ]:
# ==========================================
# 1. Validazione per K-Means (Euclideo)
# ==========================================
k_range = range(2, 20)
inertia_values = []
silhouette_scores = []

for k_val in k_range:
    kmeans_eval = TimeSeriesKMeans(
        n_clusters=k_val,
        metric="euclidean",
        random_state=222
    )
    # L'algoritmo di sktime riceve l'array 3D
    labels_eval = kmeans_eval.fit_predict(X_sktime)

    inertia_values.append(kmeans_eval.inertia_)

    # La metrica Silhouette di scikit-learn richiede array 2D
    score = silhouette_score(X_flattened, labels_eval, metric='euclidean')
    silhouette_scores.append(score)

# Visualizzazione Metodo del Gomito e Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertia_values, marker='o', color='blue')
ax1.set_title('Metodo del Gomito (Elbow Method)')
ax1.set_xlabel('Numero di Cluster (k)')
ax1.set_ylabel('Inerzia')
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(k_range, silhouette_scores, marker='o', color='orange')
ax2.set_title('Indice di Silhouette')
ax2.set_xlabel('Numero di Cluster (k)')
ax2.set_ylabel('Silhouette Score Medio')
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ==========================================
# 2. Validazione per Clustering Gerarchico
# ==========================================
# Il linkage 'ward' calcola le distanze in base alla varianza ed è coerente con l'implementazione
Z = linkage(X_flattened, method='ward')

plt.figure(figsize=(12, 7))
plt.title("Dendrogramma per la Scelta di K")
# Troncamento visivo per evitare un albero illeggibile in presenza di molti campioni
dendrogram(Z, truncate_mode='lastp', p=40)
plt.xlabel("Dimensione dei cluster / Indici dei campioni")
plt.ylabel("Distanza di Ward")
plt.show()